# 03 Data Preparation and ABT Creation

This notebook prepares the final Analytical Base Table (ABT) for the AgriCredit Resilience project.

The goal is to clean, transform, and merge the raw source tables into one machine-learning-ready dataset.

The final output will be:

`data/processed/farmer_vulnerability_abt.csv`

Main tasks:

1. Load raw CSV files
2. Clean inconsistent values
3. Handle missing values
4. Engineer predictive features
5. Merge household-level and state-level tables
6. Create the target feature: `financial_vulnerability`
7. Save and validate the final ABT

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR, PROCESSED_DATA_DIR

(PosixPath('/Users/isaacaung/Desktop/agricredit-resilience/data/raw'),
 PosixPath('/Users/isaacaung/Desktop/agricredit-resilience/data/processed'))

In [3]:
raw_files = {
    "households": "households_raw.csv",
    "agriculture": "agriculture_raw.csv",
    "coping": "coping_raw.csv",
    "market_prices": "market_prices_raw.csv",
    "meb_values": "meb_values_raw.csv",
    "assistance_coverage": "assistance_coverage_raw.csv",
}

data = {}

for name, filename in raw_files.items():
    data[name] = pd.read_csv(RAW_DATA_DIR / filename)

print("Raw data loaded successfully.")

Raw data loaded successfully.


In [4]:
for name, df in data.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

households: 1500 rows, 13 columns
agriculture: 1500 rows, 7 columns
coping: 1500 rows, 7 columns
market_prices: 14 rows, 6 columns
meb_values: 14 rows, 3 columns
assistance_coverage: 14 rows, 6 columns


## Cleaning Plan

The data understanding notebook showed several raw data quality issues.

Cleaning actions:

1. Convert money values such as `"330945 MMK"` into numeric values.
2. Standardize Yes/No values:
   - `Yes`, `yes`, `Y` → 1
   - `No`, `no`, `N` → 0
3. Standardize state/region names:
   - `sagaing` → `Sagaing`
   - `Magwe` → `Magway`
   - `Southern Shan` → `Shan South`
   - `Northern Shan` → `Shan North`
4. Fill missing `main_crop` values with `None` for non-farming households.
5. Handle missing numeric values using appropriate median values.
6. Convert `basic_needs_met` into an ordered numeric score.

In [5]:
def clean_money_column(series: pd.Series) -> pd.Series:
    """
    Convert raw money values into numeric values.

    Examples:
    420000 -> 420000
    "330945 MMK" -> 330945
    missing values -> NaN
    """
    cleaned = (
        series
        .astype("string")
        .str.replace("MMK", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    return pd.to_numeric(cleaned, errors="coerce")

In [6]:
def clean_yes_no_column(series: pd.Series) -> pd.Series:
    """
    Convert messy Yes/No values into 1/0.

    Yes, yes, Y -> 1
    No, no, N -> 0
    Unknown -> NaN
    """
    cleaned = series.astype("string").str.strip().str.lower()

    mapping = {
        "yes": 1,
        "y": 1,
        "no": 0,
        "n": 0,
    }

    return cleaned.map(mapping)

In [7]:
def standardize_state_region(series: pd.Series) -> pd.Series:
    """
    Standardize inconsistent state/region names.
    """
    cleaned = series.astype("string").str.strip()

    mapping = {
        "sagaing": "Sagaing",
        "Magwe": "Magway",
        "Southern Shan": "Shan South",
        "Northern Shan": "Shan North",
    }

    return cleaned.replace(mapping)

In [8]:
def clean_basic_needs(series: pd.Series) -> pd.Series:
    """
    Convert basic needs coverage into an ordered numeric score.

    Higher score means better basic needs coverage.
    """
    cleaned = series.astype("string").str.strip().str.lower()

    mapping = {
        "yes fully": 3,
        "yes mostly": 2,
        "only some": 1,
        "no": 0,
    }

    return cleaned.map(mapping)

## Clean Household Table

The household table contains the main household-level information, including location, household size, income, debt, savings, market access, and vulnerability-related household characteristics.

Cleaning actions:

1. Standardize inconsistent `state_region` names.
2. Convert `monthly_income`, `total_debt`, and `monthly_debt_repayment` into numeric values.
3. Convert Yes/No columns into binary values:
   - `has_debt`
   - `market_access`
   - `female_headed_household`
   - `disability_present`
4. Fill missing `monthly_income` values using the median income.
5. Treat unknown `market_access` values conservatively as no market access.

In [9]:
households = data["households"].copy()

households["state_region"] = standardize_state_region(households["state_region"])

households["monthly_income"] = clean_money_column(households["monthly_income"])
households["total_debt"] = clean_money_column(households["total_debt"])
households["monthly_debt_repayment"] = clean_money_column(households["monthly_debt_repayment"])

households["has_debt"] = clean_yes_no_column(households["has_debt"])
households["market_access"] = clean_yes_no_column(households["market_access"])
households["female_headed_household"] = clean_yes_no_column(households["female_headed_household"])
households["disability_present"] = clean_yes_no_column(households["disability_present"])

# Fill missing monthly income using median income
households["monthly_income"] = households["monthly_income"].fillna(
    households["monthly_income"].median()
)

# Treat unknown market access conservatively as no access
households["market_access"] = households["market_access"].fillna(0)

households.head()

,household_id,state_region,township,household_size,monthly_income,has_debt,total_debt,monthly_debt_repayment,savings_duration_weeks,market_access,female_headed_household,disability_present,displacement_status
0,HH00001,Magway,Magway,6,610149.0,1,851624,150329,1,1.0,0,0,IDP
1,HH00002,Ayeyarwady,Pathein,8,284061.0,1,801819,95454,4,0.0,0,0,Resident
2,HH00003,Shan North,Lashio,4,50000.0,0,0,0,2,0.0,0,0,Resident
3,HH00004,Kayin,Hpa-An,4,101947.0,1,667989,60245,2,1.0,0,0,Resident
4,HH00005,Kayin,Myawaddy,6,675893.0,1,593253,81094,1,0.0,0,0,Returnee


In [10]:
households_check = {
    "monthly_income_missing": households["monthly_income"].isna().sum(),
    "has_debt_values": households["has_debt"].value_counts(dropna=False).to_dict(),
    "market_access_values": households["market_access"].value_counts(dropna=False).to_dict(),
    "state_region_unique_count": households["state_region"].nunique(),
}

households_check

{'monthly_income_missing': np.int64(0),
 'has_debt_values': {1: 1008, 0: 492},
 'market_access_values': {1.0: 927, 0.0: 573},
 'state_region_unique_count': 14}

### Observation

The household table cleaning was successful. Missing monthly income values were filled, and the `has_debt` and `market_access` columns were converted into binary values. The `state_region` column now contains 14 unique state/region names, which means the inconsistent names found during data understanding were standardized correctly.

This cleaned household table is now ready to be merged with other household-level and state-level tables.

## Clean Agriculture Table

The agriculture table contains farming-related information such as crop type, farm size, irrigation access, fertilizer cost, and crop damage.

Cleaning actions:

1. Convert `fertilizer_cost` into numeric values.
2. Convert Yes/No columns into binary values.
3. Fill missing `main_crop` values with `None`.
4. Fill missing `fertilizer_cost` values with the median fertilizer cost.

In [11]:
agriculture = data["agriculture"].copy()

agriculture["fertilizer_cost"] = clean_money_column(agriculture["fertilizer_cost"])

agriculture["is_farming_household"] = clean_yes_no_column(agriculture["is_farming_household"])
agriculture["irrigation_access"] = clean_yes_no_column(agriculture["irrigation_access"])
agriculture["crop_damage_recent"] = clean_yes_no_column(agriculture["crop_damage_recent"])

# Missing crop values represent non-farming households
agriculture["main_crop"] = agriculture["main_crop"].fillna("None")

# Fill missing fertilizer cost using median fertilizer cost
agriculture["fertilizer_cost"] = agriculture["fertilizer_cost"].fillna(
    agriculture["fertilizer_cost"].median()
)

agriculture.head()

,household_id,is_farming_household,main_crop,farm_size_acres,irrigation_access,fertilizer_cost,crop_damage_recent
0,HH00001,1,Rice,3.80,0,88356.0,0
1,HH00002,1,Rice,2.90,0,236635.0,0
2,HH00003,1,Rice,0.63,0,185682.0,1
3,HH00004,0,None,0.00,0,0.0,0
4,HH00005,0,None,0.00,0,0.0,0


In [12]:
agriculture_check = {
    "main_crop_missing": agriculture["main_crop"].isna().sum(),
    "fertilizer_cost_missing": agriculture["fertilizer_cost"].isna().sum(),
    "is_farming_household_values": agriculture["is_farming_household"].value_counts(dropna=False).to_dict(),
    "irrigation_access_values": agriculture["irrigation_access"].value_counts(dropna=False).to_dict(),
    "crop_damage_recent_values": agriculture["crop_damage_recent"].value_counts(dropna=False).to_dict(),
    "main_crop_values": agriculture["main_crop"].value_counts(dropna=False).to_dict(),
}

agriculture_check

{'main_crop_missing': np.int64(0),
 'fertilizer_cost_missing': np.int64(0),
 'is_farming_household_values': {1: 1142, 0: 358},
 'irrigation_access_values': {0: 1034, 1: 466},
 'crop_damage_recent_values': {0: 1194, 1: 306},
 'main_crop_values': {'Rice': 479,
  'None': 358,
  'Pulses': 192,
  'Maize': 143,
  'Oilseed': 131,
  'Mixed': 110,
  'Vegetables': 87}}

### Observation

The agriculture table cleaning was successful. Missing `main_crop` values were converted to `None`, which correctly represents non-farming households. Missing `fertilizer_cost` values were filled using the median value. The Yes/No columns were converted into binary values.

The cleaned agriculture table is now ready to be merged with the household table using `household_id`.

## Clean Coping Table

The coping table contains information about household food-related coping strategies and basic needs coverage.

Cleaning actions:

1. Convert coping strategy day columns into numeric values.
2. Fill missing coping strategy values using median values.
3. Convert `basic_needs_met` into an ordered numeric score.
4. Create a coping strategy score later during feature engineering.

In [13]:
coping = data["coping"].copy()

coping_day_cols = [
    "less_preferred_food_days",
    "borrowed_food_days",
    "reduced_meals_days",
    "reduced_portion_days",
    "adults_reduced_for_children_days",
]

for col in coping_day_cols:
    coping[col] = pd.to_numeric(coping[col], errors="coerce")
    coping[col] = coping[col].fillna(coping[col].median())

coping["basic_needs_score"] = clean_basic_needs(coping["basic_needs_met"])

coping.head()

,household_id,less_preferred_food_days,borrowed_food_days,reduced_meals_days,reduced_portion_days,adults_reduced_for_children_days,basic_needs_met,basic_needs_score
0,HH00001,0.0,0.0,0.0,2.0,0.0,Yes mostly,2
1,HH00002,6.0,0.0,2.0,6.0,1.0,Yes mostly,2
2,HH00003,5.0,2.0,1.0,3.0,3.0,Only some,1
3,HH00004,1.0,1.0,2.0,1.0,3.0,Yes mostly,2
4,HH00005,7.0,2.0,5.0,3.0,1.0,Only some,1


In [14]:
coping_check = {
    "less_preferred_food_days_missing": coping["less_preferred_food_days"].isna().sum(),
    "borrowed_food_days_missing": coping["borrowed_food_days"].isna().sum(),
    "reduced_meals_days_missing": coping["reduced_meals_days"].isna().sum(),
    "reduced_portion_days_missing": coping["reduced_portion_days"].isna().sum(),
    "adults_reduced_for_children_days_missing": coping["adults_reduced_for_children_days"].isna().sum(),
    "basic_needs_score_missing": coping["basic_needs_score"].isna().sum(),
    "basic_needs_score_values": coping["basic_needs_score"].value_counts(dropna=False).to_dict(),
}

coping_check

{'less_preferred_food_days_missing': np.int64(0),
 'borrowed_food_days_missing': np.int64(0),
 'reduced_meals_days_missing': np.int64(0),
 'reduced_portion_days_missing': np.int64(0),
 'adults_reduced_for_children_days_missing': np.int64(0),
 'basic_needs_score_missing': np.int64(0),
 'basic_needs_score_values': {2: 534, 1: 472, 3: 275, 0: 219}}

### Observation

The coping table cleaning was successful. Missing values in the coping strategy day columns were filled using median values. The `basic_needs_met` column was converted into an ordered numeric score, where a higher score means better basic needs coverage.

This cleaned coping table is now ready for feature engineering, especially the creation of an rCSI-style coping strategy score.

## Clean State-Level Tables

The state-level tables contain contextual information about market prices, MEB values, and assistance coverage.

Cleaning actions:

1. Standardize `state_region` names.
2. Convert numeric columns into proper numeric values.
3. Convert `market_disruption_level` into an ordered score:
   - `Low` → 0
   - `Medium` → 1
   - `High` → 2
4. Prepare these tables for merging with the household-level ABT using `state_region`.

In [15]:
market_prices = data["market_prices"].copy()

market_prices["state_region"] = standardize_state_region(market_prices["state_region"])

market_numeric_cols = [
    "basic_food_basket_change_1y",
    "rice_price_change_1y",
    "fuel_price_change_1y",
]

for col in market_numeric_cols:
    market_prices[col] = pd.to_numeric(market_prices[col], errors="coerce")

market_disruption_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
}

market_prices["market_disruption_score"] = market_prices["market_disruption_level"].map(
    market_disruption_mapping
)

market_prices.head()

,state_region,month,basic_food_basket_change_1y,rice_price_change_1y,fuel_price_change_1y,market_disruption_level,market_disruption_score
0,Sagaing,2025-04,34.4,10.9,34.3,Medium,1
1,Magway,2025-04,61.4,11.4,8.6,Medium,1
2,Mandalay,2025-04,11.3,9.2,35.3,Low,0
3,Chin,2025-04,39.8,12.5,25.0,Medium,1
4,Kachin,2025-04,15.2,18.4,1.1,Low,0


In [16]:
meb_values = data["meb_values"].copy()

meb_values["state_region"] = standardize_state_region(meb_values["state_region"])

meb_values["mpca_per_person"] = clean_money_column(meb_values["mpca_per_person"])
meb_values["cash_food_assistance_per_person"] = clean_money_column(
    meb_values["cash_food_assistance_per_person"]
)

meb_values.head()

,state_region,mpca_per_person,cash_food_assistance_per_person
0,Bago,55000,35000
1,Chin,95000,65000
2,Kachin,65000,50000
3,Kayah,80000,55000
4,Kayin,65000,40000


In [17]:
assistance_coverage = data["assistance_coverage"].copy()

assistance_coverage["state_region"] = standardize_state_region(
    assistance_coverage["state_region"]
)

assistance_numeric_cols = [
    "people_targeted",
    "people_reached",
    "coverage_rate",
    "response_gap",
    "number_of_partners_active",
]

for col in assistance_numeric_cols:
    assistance_coverage[col] = pd.to_numeric(assistance_coverage[col], errors="coerce")

assistance_coverage.head()

,state_region,people_targeted,people_reached,coverage_rate,response_gap,number_of_partners_active
0,Sagaing,178672,122516,0.686,56156,12
1,Magway,24770,9411,0.380,15359,21
2,Mandalay,147038,87058,0.592,59980,26
3,Chin,104011,39085,0.376,64926,10
4,Kachin,59479,21163,0.356,38316,20


In [18]:
state_level_check = {
    "market_prices_state_count": market_prices["state_region"].nunique(),
    "meb_values_state_count": meb_values["state_region"].nunique(),
    "assistance_coverage_state_count": assistance_coverage["state_region"].nunique(),
    "market_disruption_score_missing": market_prices["market_disruption_score"].isna().sum(),
    "mpca_per_person_missing": meb_values["mpca_per_person"].isna().sum(),
    "coverage_rate_missing": assistance_coverage["coverage_rate"].isna().sum(),
}

state_level_check

{'market_prices_state_count': 14,
 'meb_values_state_count': 14,
 'assistance_coverage_state_count': 14,
 'market_disruption_score_missing': np.int64(0),
 'mpca_per_person_missing': np.int64(0),
 'coverage_rate_missing': np.int64(0)}

### Observation

The state-level tables were cleaned successfully. Each table contains 14 unique state/region names, which means they are ready to be merged into the household-level ABT using `state_region`.

The `market_disruption_level` column was converted into an ordered numeric score, where higher values represent more serious market disruption. Numeric values in the MEB and assistance coverage tables were also prepared for later feature engineering.

## Feature Engineering

After cleaning the raw tables, the next step is to create new predictive features.

Feature engineering creates more meaningful variables from existing columns. These features will help the machine learning models identify patterns related to household financial vulnerability.

New features to create:

1. `debt_to_income_ratio`
2. `monthly_debt_repayment_ratio`
3. `rCSI_score`
4. `market_pressure_score`
5. `household_meb_requirement`
6. `income_to_meb_ratio`
7. `meb_gap`
8. `coverage_gap_ratio`

In [19]:
households_features = households.copy()

# Avoid division by zero
households_features["debt_to_income_ratio"] = (
    households_features["total_debt"] / households_features["monthly_income"]
)

households_features["monthly_debt_repayment_ratio"] = (
    households_features["monthly_debt_repayment"] / households_features["monthly_income"]
)

# Replace infinite values if any
households_features["debt_to_income_ratio"] = households_features["debt_to_income_ratio"].replace(
    [np.inf, -np.inf], np.nan
)

households_features["monthly_debt_repayment_ratio"] = households_features[
    "monthly_debt_repayment_ratio"
].replace([np.inf, -np.inf], np.nan)

# Fill any remaining missing ratio values with 0
households_features["debt_to_income_ratio"] = households_features["debt_to_income_ratio"].fillna(0)
households_features["monthly_debt_repayment_ratio"] = households_features[
    "monthly_debt_repayment_ratio"
].fillna(0)

households_features[[
    "household_id",
    "monthly_income",
    "total_debt",
    "monthly_debt_repayment",
    "debt_to_income_ratio",
    "monthly_debt_repayment_ratio",
]].head()

,household_id,monthly_income,total_debt,monthly_debt_repayment,debt_to_income_ratio,monthly_debt_repayment_ratio
0,HH00001,610149.0,851624,150329,1.395764,0.246381
1,HH00002,284061.0,801819,95454,2.8227,0.336033
2,HH00003,50000.0,0,0,0.0,0.0
3,HH00004,101947.0,667989,60245,6.552316,0.590944
4,HH00005,675893.0,593253,81094,0.877732,0.119981


In [20]:
households_features[[
    "debt_to_income_ratio",
    "monthly_debt_repayment_ratio"
]].describe()

,debt_to_income_ratio,monthly_debt_repayment_ratio
count,1500.0,1500.0
mean,1.369066,0.152752
std,2.060826,0.232416
min,0.0,0.0
25%,0.0,0.0
50%,0.954637,0.098998
75%,1.887246,0.214159
max,31.27772,3.03662


### Observation

The household financial features were created successfully. The `debt_to_income_ratio` compares total debt with monthly income, while the `monthly_debt_repayment_ratio` compares monthly debt repayment with monthly income. These features are useful because households with higher debt pressure may be more financially vulnerable.

In [21]:
coping_features = coping.copy()

# rCSI-style weighted coping score
coping_features["rCSI_score"] = (
    coping_features["less_preferred_food_days"] * 1
    + coping_features["borrowed_food_days"] * 2
    + coping_features["reduced_meals_days"] * 1
    + coping_features["reduced_portion_days"] * 1
    + coping_features["adults_reduced_for_children_days"] * 3
)

coping_features[[
    "household_id",
    "less_preferred_food_days",
    "borrowed_food_days",
    "reduced_meals_days",
    "reduced_portion_days",
    "adults_reduced_for_children_days",
    "rCSI_score",
    "basic_needs_score",
]].head()

,household_id,less_preferred_food_days,borrowed_food_days,reduced_meals_days,reduced_portion_days,adults_reduced_for_children_days,rCSI_score,basic_needs_score
0,HH00001,0.0,0.0,0.0,2.0,0.0,2.0,2
1,HH00002,6.0,0.0,2.0,6.0,1.0,17.0,2
2,HH00003,5.0,2.0,1.0,3.0,3.0,22.0,1
3,HH00004,1.0,1.0,2.0,1.0,3.0,15.0,2
4,HH00005,7.0,2.0,5.0,3.0,1.0,22.0,1


In [22]:
coping_features[["rCSI_score", "basic_needs_score"]].describe()

,rCSI_score,basic_needs_score
count,1500.000000,1500.000000
mean,19.905333,1.576667
std,6.463453,0.950469
min,2.000000,0.000000
25%,15.000000,1.000000
50%,20.000000,2.000000
75%,24.000000,2.000000
max,39.000000,3.000000


### Observation

The `rCSI_score` feature was created from food-related coping strategy columns. A higher score means the household uses more coping strategies or uses more severe coping strategies. This can indicate higher food insecurity and financial vulnerability.

In [23]:
market_features = market_prices.copy()

market_features["market_pressure_score"] = (
    market_features["basic_food_basket_change_1y"]
    + market_features["rice_price_change_1y"]
    + market_features["fuel_price_change_1y"]
    + (market_features["market_disruption_score"] * 10)
)

market_features[[
    "state_region",
    "basic_food_basket_change_1y",
    "rice_price_change_1y",
    "fuel_price_change_1y",
    "market_disruption_score",
    "market_pressure_score",
]].head()

,state_region,basic_food_basket_change_1y,rice_price_change_1y,fuel_price_change_1y,market_disruption_score,market_pressure_score
0,Sagaing,34.4,10.9,34.3,1,89.6
1,Magway,61.4,11.4,8.6,1,91.4
2,Mandalay,11.3,9.2,35.3,0,55.8
3,Chin,39.8,12.5,25.0,1,87.3
4,Kachin,15.2,18.4,1.1,0,34.7


In [24]:
market_features[["market_pressure_score"]].describe()

,market_pressure_score
count,14.000000
mean,75.650000
std,17.419872
min,34.700000
25%,67.400000
50%,78.850000
75%,89.025000
max,102.000000


### Observation

The `market_pressure_score` combines food basket price change, rice price change, fuel price change, and market disruption. A higher score means households in that state/region may face stronger market pressure.

In [25]:
assistance_features = assistance_coverage.copy()

assistance_features["coverage_gap_ratio"] = 1 - assistance_features["coverage_rate"]

assistance_features[[
    "state_region",
    "coverage_rate",
    "coverage_gap_ratio",
    "response_gap",
    "number_of_partners_active",
]].head()

,state_region,coverage_rate,coverage_gap_ratio,response_gap,number_of_partners_active
0,Sagaing,0.686,0.314,56156,12
1,Magway,0.380,0.620,15359,21
2,Mandalay,0.592,0.408,59980,26
3,Chin,0.376,0.624,64926,10
4,Kachin,0.356,0.644,38316,20


In [26]:
assistance_features[["coverage_rate", "coverage_gap_ratio"]].describe()

,coverage_rate,coverage_gap_ratio
count,14.000000,14.000000
mean,0.455857,0.544143
std,0.145970,0.145970
min,0.256000,0.294000
25%,0.361750,0.412500
50%,0.383500,0.616500
75%,0.587500,0.638250
max,0.706000,0.744000


### Observation

The `coverage_gap_ratio` feature was created from the assistance coverage rate. A higher coverage gap means a larger share of targeted people were not reached. This can provide useful state-level context for household vulnerability.

## Merge Tables into the ABT

After cleaning and feature engineering, the next step is to merge the tables into one Analytical Base Table.

The household-level tables are merged using `household_id`:

- `households_features`
- `agriculture`
- `coping_features`

The state-level tables are merged using `state_region`:

- `market_features`
- `meb_values`
- `assistance_features`

The final ABT should contain one row per household.

In [27]:
abt = households_features.merge(
    agriculture,
    on="household_id",
    how="left"
)

abt = abt.merge(
    coping_features,
    on="household_id",
    how="left"
)

abt.shape

(1500, 29)

In [28]:
abt = abt.merge(
    market_features,
    on="state_region",
    how="left"
)

abt = abt.merge(
    meb_values,
    on="state_region",
    how="left"
)

abt = abt.merge(
    assistance_features,
    on="state_region",
    how="left"
)

abt.shape

(1500, 44)

In [29]:
merge_check = {
    "abt_rows": abt.shape[0],
    "abt_columns": abt.shape[1],
    "unique_household_ids": abt["household_id"].nunique(),
    "duplicate_household_ids": abt["household_id"].duplicated().sum(),
    "missing_market_pressure_score": abt["market_pressure_score"].isna().sum(),
    "missing_mpca_per_person": abt["mpca_per_person"].isna().sum(),
    "missing_coverage_gap_ratio": abt["coverage_gap_ratio"].isna().sum(),
}

merge_check

{'abt_rows': 1500,
 'abt_columns': 44,
 'unique_household_ids': 1500,
 'duplicate_household_ids': np.int64(0),
 'missing_market_pressure_score': np.int64(0),
 'missing_mpca_per_person': np.int64(0),
 'missing_coverage_gap_ratio': np.int64(0)}

In [30]:
abt.head()

,household_id,state_region,township,household_size,monthly_income,has_debt,total_debt,monthly_debt_repayment,savings_duration_weeks,market_access,...,market_disruption_score,market_pressure_score,mpca_per_person,cash_food_assistance_per_person,people_targeted,people_reached,coverage_rate,response_gap,number_of_partners_active,coverage_gap_ratio
0,HH00001,Magway,Magway,6,610149.0,1,851624,150329,1,1.0,...,1,91.4,55000,30000,24770,9411,0.380,15359,21,0.620
1,HH00002,Ayeyarwady,Pathein,8,284061.0,1,801819,95454,4,0.0,...,1,70.9,55000,35000,97686,41448,0.424,56238,21,0.576
2,HH00003,Shan North,Lashio,4,50000.0,0,0,0,2,0.0,...,2,78.5,60000,50000,23931,14516,0.607,9415,17,0.393
3,HH00004,Kayin,Hpa-An,4,101947.0,1,667989,60245,2,1.0,...,1,66.5,65000,40000,208894,80818,0.387,128076,20,0.613
4,HH00005,Kayin,Myawaddy,6,675893.0,1,593253,81094,1,0.0,...,1,66.5,65000,40000,208894,80818,0.387,128076,20,0.613


### Observation

The merge was successful. The final ABT currently contains 1,500 rows and 44 columns. The number of unique household IDs is also 1,500, and there are no duplicate household IDs.

This confirms that the household-level tables were merged correctly using `household_id`, and the state-level tables were merged correctly using `state_region`. Important state-level features such as `market_pressure_score`, `mpca_per_person`, and `coverage_gap_ratio` have no missing values after merging.

## Create MEB-Related Features

The MEB-related features compare household income with the estimated minimum expenditure requirement.

New features:

1. `household_meb_requirement`
   - estimated minimum expenditure requirement for the household

2. `income_to_meb_ratio`
   - compares monthly income with the household MEB requirement

3. `meb_gap`
   - shows how much income is below or above the household MEB requirement

In [31]:
abt["household_meb_requirement"] = (
    abt["household_size"] * abt["mpca_per_person"]
)

abt["income_to_meb_ratio"] = (
    abt["monthly_income"] / abt["household_meb_requirement"]
)

abt["meb_gap"] = (
    abt["monthly_income"] - abt["household_meb_requirement"]
)

abt[[
    "household_id",
    "household_size",
    "monthly_income",
    "mpca_per_person",
    "household_meb_requirement",
    "income_to_meb_ratio",
    "meb_gap",
]].head()

,household_id,household_size,monthly_income,mpca_per_person,household_meb_requirement,income_to_meb_ratio,meb_gap
0,HH00001,6,610149.0,55000,330000,1.848936,280149.0
1,HH00002,8,284061.0,55000,440000,0.645593,-155939.0
2,HH00003,4,50000.0,60000,240000,0.208333,-190000.0
3,HH00004,4,101947.0,65000,260000,0.392104,-158053.0
4,HH00005,6,675893.0,65000,390000,1.733059,285893.0


In [32]:
abt[[
    "household_meb_requirement",
    "income_to_meb_ratio",
    "meb_gap",
]].describe()

,household_meb_requirement,income_to_meb_ratio,meb_gap
count,1500.0,1500.0,1500.0
mean,290050.0,1.879287,131380.494667
std,130053.668939,1.615801,211721.873048
min,55000.0,0.113636,-603163.0
25%,220000.0,0.977781,-5856.5
50%,275000.0,1.489503,131902.0
75%,385000.0,2.217638,268473.75
max,855000.0,17.532382,909281.0


In [33]:
meb_feature_check = {
    "household_meb_requirement_missing": abt["household_meb_requirement"].isna().sum(),
    "income_to_meb_ratio_missing": abt["income_to_meb_ratio"].isna().sum(),
    "meb_gap_missing": abt["meb_gap"].isna().sum(),
}

meb_feature_check

{'household_meb_requirement_missing': np.int64(0),
 'income_to_meb_ratio_missing': np.int64(0),
 'meb_gap_missing': np.int64(0)}

### Observation

The MEB-related features were created successfully. The `household_meb_requirement` estimates the minimum expenditure requirement based on household size and state-level MPCA value.

The `income_to_meb_ratio` compares household income with the estimated minimum requirement. A value below 1 means the household income is below the estimated minimum requirement. The `meb_gap` shows the income surplus or shortfall compared with the household MEB requirement.

These features are important for identifying household financial vulnerability.

## Create Target Feature: Financial Vulnerability

The project does not use real loan default data. Instead, it creates a proxy target called `financial_vulnerability`.

This target is based on several vulnerability indicators:

1. Household income is below the estimated MEB requirement.
2. Debt is high compared with income.
3. Monthly debt repayment pressure is high.
4. rCSI score is high.
5. Basic needs coverage is poor.
6. Household has no market access.
7. Household experienced recent crop damage.
8. Household lives in a high market pressure area.

The target feature is binary:

- `1` = financially vulnerable
- `0` = not financially vulnerable

In [34]:
abt["low_income_flag"] = (abt["income_to_meb_ratio"] < 1).astype(int)

abt["high_debt_flag"] = (abt["debt_to_income_ratio"] > 1).astype(int)

abt["high_repayment_pressure_flag"] = (
    abt["monthly_debt_repayment_ratio"] > 0.20
).astype(int)

abt["high_rcsi_flag"] = (abt["rCSI_score"] >= 20).astype(int)

abt["poor_basic_needs_flag"] = (abt["basic_needs_score"] <= 1).astype(int)

abt["no_market_access_flag"] = (abt["market_access"] == 0).astype(int)

abt["crop_damage_flag"] = (abt["crop_damage_recent"] == 1).astype(int)

abt["high_market_pressure_flag"] = (
    abt["market_pressure_score"] >= abt["market_pressure_score"].median()
).astype(int)

abt[[
    "household_id",
    "low_income_flag",
    "high_debt_flag",
    "high_repayment_pressure_flag",
    "high_rcsi_flag",
    "poor_basic_needs_flag",
    "no_market_access_flag",
    "crop_damage_flag",
    "high_market_pressure_flag",
]].head()

,household_id,low_income_flag,high_debt_flag,high_repayment_pressure_flag,high_rcsi_flag,poor_basic_needs_flag,no_market_access_flag,crop_damage_flag,high_market_pressure_flag
0,HH00001,0,1,1,0,0,0,0,1
1,HH00002,1,1,1,0,0,1,0,0
2,HH00003,1,0,0,1,1,1,1,0
3,HH00004,1,1,1,0,0,0,0,0
4,HH00005,0,0,0,1,1,1,0,0


In [35]:
vulnerability_flags = [
    "low_income_flag",
    "high_debt_flag",
    "high_repayment_pressure_flag",
    "high_rcsi_flag",
    "poor_basic_needs_flag",
    "no_market_access_flag",
    "crop_damage_flag",
    "high_market_pressure_flag",
]

abt["vulnerability_score"] = abt[vulnerability_flags].sum(axis=1)

abt[[
    "household_id",
    "vulnerability_score",
    *vulnerability_flags,
]].head()

,household_id,vulnerability_score,low_income_flag,high_debt_flag,high_repayment_pressure_flag,high_rcsi_flag,poor_basic_needs_flag,no_market_access_flag,crop_damage_flag,high_market_pressure_flag
0,HH00001,3,0,1,1,0,0,0,0,1
1,HH00002,4,1,1,1,0,0,1,0,0
2,HH00003,5,1,0,0,1,1,1,1,0
3,HH00004,3,1,1,1,0,0,0,0,0
4,HH00005,3,0,0,0,1,1,1,0,0


In [36]:
abt["financial_vulnerability"] = (abt["vulnerability_score"] >= 3).astype(int)

abt[[
    "household_id",
    "vulnerability_score",
    "financial_vulnerability",
]].head()

,household_id,vulnerability_score,financial_vulnerability
0,HH00001,3,1
1,HH00002,4,1
2,HH00003,5,1
3,HH00004,3,1
4,HH00005,3,1


In [37]:
target_distribution = abt["financial_vulnerability"].value_counts().sort_index()

target_distribution

financial_vulnerability
0    555
1    945
Name: count, dtype: int64

In [38]:
target_distribution_percent = (
    abt["financial_vulnerability"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
).round(2)

target_distribution_percent

financial_vulnerability
0    37.0
1    63.0
Name: proportion, dtype: float64

In [39]:
abt["vulnerability_score"].value_counts().sort_index()

vulnerability_score
0     30
1    154
2    371
3    391
4    281
5    176
6     81
7     15
8      1
Name: count, dtype: int64

### Observation

The target feature `financial_vulnerability` was created as a proxy target because real loan default data is not available. The target is based on multiple vulnerability indicators, including low income compared with MEB requirement, high debt pressure, high coping strategy score, poor basic needs coverage, lack of market access, crop damage, and high market pressure.

The `vulnerability_score` counts how many risk indicators apply to each household. Households with a vulnerability score of 3 or higher are labeled as financially vulnerable.

This target will be used later for Decision Tree and Logistic Regression modeling.

## Select Final ABT Columns

The ABT currently contains many columns from the original raw tables, cleaned tables, engineered features, and target creation process.

For modeling, we will keep:

1. Identifier and location columns
2. Descriptive features
3. Engineered features
4. Target feature

The final target feature is:

`financial_vulnerability`

In [40]:
final_columns = [
    # ID and location
    "household_id",
    "state_region",
    "township",

    # Household demographic and vulnerability characteristics
    "household_size",
    "female_headed_household",
    "disability_present",
    "displacement_status",

    # Financial features
    "monthly_income",
    "has_debt",
    "total_debt",
    "monthly_debt_repayment",
    "savings_duration_weeks",
    "debt_to_income_ratio",
    "monthly_debt_repayment_ratio",

    # Market access
    "market_access",

    # Agriculture features
    "is_farming_household",
    "main_crop",
    "farm_size_acres",
    "irrigation_access",
    "fertilizer_cost",
    "crop_damage_recent",

    # Coping and basic needs features
    "less_preferred_food_days",
    "borrowed_food_days",
    "reduced_meals_days",
    "reduced_portion_days",
    "adults_reduced_for_children_days",
    "rCSI_score",
    "basic_needs_score",

    # Market context features
    "basic_food_basket_change_1y",
    "rice_price_change_1y",
    "fuel_price_change_1y",
    "market_disruption_score",
    "market_pressure_score",

    # MEB features
    "mpca_per_person",
    "cash_food_assistance_per_person",
    "household_meb_requirement",
    "income_to_meb_ratio",
    "meb_gap",

    # Assistance coverage features
    "coverage_rate",
    "coverage_gap_ratio",
    "response_gap",
    "number_of_partners_active",

    # Target construction fields
    "vulnerability_score",

    # Target feature
    "financial_vulnerability",
]

final_abt = abt[final_columns].copy()

final_abt.head()

,household_id,state_region,township,household_size,female_headed_household,disability_present,displacement_status,monthly_income,has_debt,total_debt,...,cash_food_assistance_per_person,household_meb_requirement,income_to_meb_ratio,meb_gap,coverage_rate,coverage_gap_ratio,response_gap,number_of_partners_active,vulnerability_score,financial_vulnerability
0,HH00001,Magway,Magway,6,0,0,IDP,610149.0,1,851624,...,30000,330000,1.848936,280149.0,0.380,0.620,15359,21,3,1
1,HH00002,Ayeyarwady,Pathein,8,0,0,Resident,284061.0,1,801819,...,35000,440000,0.645593,-155939.0,0.424,0.576,56238,21,4,1
2,HH00003,Shan North,Lashio,4,0,0,Resident,50000.0,0,0,...,50000,240000,0.208333,-190000.0,0.607,0.393,9415,17,5,1
3,HH00004,Kayin,Hpa-An,4,0,0,Resident,101947.0,1,667989,...,40000,260000,0.392104,-158053.0,0.387,0.613,128076,20,3,1
4,HH00005,Kayin,Myawaddy,6,0,0,Returnee,675893.0,1,593253,...,40000,390000,1.733059,285893.0,0.387,0.613,128076,20,3,1


In [41]:
final_abt.shape

(1500, 44)

In [42]:
final_missing = final_abt.isna().sum().sort_values(ascending=False)

final_missing[final_missing > 0]

Series([], dtype: int64)

In [43]:
final_abt["financial_vulnerability"].value_counts().sort_index()

financial_vulnerability
0    555
1    945
Name: count, dtype: int64

In [44]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / "farmer_vulnerability_abt.csv"

final_abt.to_csv(output_path, index=False)

output_path

PosixPath('/Users/isaacaung/Desktop/agricredit-resilience/data/processed/farmer_vulnerability_abt.csv')

In [45]:
saved_abt = pd.read_csv(output_path)

validation_check = {
    "saved_rows": saved_abt.shape[0],
    "saved_columns": saved_abt.shape[1],
    "unique_household_ids": saved_abt["household_id"].nunique(),
    "duplicate_household_ids": saved_abt["household_id"].duplicated().sum(),
    "target_values": saved_abt["financial_vulnerability"].value_counts().sort_index().to_dict(),
}

validation_check

{'saved_rows': 1500,
 'saved_columns': 44,
 'unique_household_ids': 1500,
 'duplicate_household_ids': np.int64(0),
 'target_values': {0: 555, 1: 945}}

## Final ABT Summary

The final ABT was created successfully and saved as:

`data/processed/farmer_vulnerability_abt.csv`

The final ABT contains one row per household. It includes household-level features, agriculture features, coping strategy features, state-level market pressure features, MEB-related features, assistance coverage features, and the target feature `financial_vulnerability`.

The final target distribution contains both classes, so the dataset is ready for the next stage: model training with Decision Tree and Logistic Regression.